# 02 - Clean monthly panel

This notebook converts the raw long-format Banco Central do Brasil SGS data downloaded in `01_download_bcb_sgs.ipynb` into a clean monthly wide panel.

The output panel has one row per month and one column per series. It will be the shared input for the descriptive plots and local projection estimates later in the project.

## Imports and paths

We load the core Python packages used throughout the project and import the transformation helpers from `src/transforms.py`. The path block is written so the notebook can run from either the project root or the `notebooks/` directory.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.transforms import (
    add_credit_shares,
    add_growth_rates,
    add_log_credit_variables,
    add_policy_variables,
    add_state_variables,
    ensure_datetime,
    first_last_nonmissing,
    missing_summary,
    monthly_panel_from_long,
)

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "bcb_sgs_all_long.csv"
DICTIONARY_FILE = PROJECT_ROOT / "data" / "series_dictionary.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "brazil_credit_monthly_panel.csv"

In [ ]:

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:

from src.transforms import (
    add_credit_shares,
    add_growth_rates,
    add_log_credit_variables,
    add_policy_variables,
    add_state_variables,
    ensure_datetime,
    first_last_nonmissing,
    missing_summary,
    monthly_panel_from_long,
)


In [ ]:

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "bcb_sgs_all_long.csv"
DICTIONARY_FILE = PROJECT_ROOT / "data" / "series_dictionary.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "brazil_credit_monthly_panel.csv"

## Load raw data and series dictionary

The raw file is a tidy, long-format table with one observation per date-series pair. The series dictionary gives the manually verified SGS code, frequency, and project name for each series, which we use below when aggregating daily series to months.

In [ ]:
raw = pd.read_csv(RAW_FILE)
series_dictionary = pd.read_csv(DICTIONARY_FILE)

display(raw.head())
display(series_dictionary.head())

print(f"Raw observations: {len(raw):,}")
print(f"Raw series in file: {raw['series'].nunique():,}")
print(f"Series in dictionary: {len(series_dictionary):,}")

## Basic raw-data checks

Before reshaping, we verify that the expected long-format columns are present, parse the date column, and summarize coverage by series. These checks help catch failed downloads, malformed dates, or unexpected missing values before they become harder to diagnose in wide format.

In [ ]:
required_columns = {"date", "value", "series"}
missing_columns = required_columns.difference(raw.columns)
if missing_columns:
    raise ValueError(f"Raw data is missing required columns: {sorted(missing_columns)}")

raw = ensure_datetime(raw, date_col="date")

print("Missing values by raw column:")
display(raw.isna().sum().rename("missing_count").to_frame())

date_range_by_series = (
    raw.groupby("series")
    .agg(first_date=("date", "min"), last_date=("date", "max"))
    .sort_index()
)
display(date_range_by_series)

observations_by_series = (
    raw.groupby("series")
    .size()
    .rename("observations")
    .sort_values(ascending=False)
    .to_frame()
)
display(observations_by_series)

## Convert to monthly panel

The empirical analysis is monthly, so every series needs to be aligned to a common monthly index. Daily series are converted to monthly averages, while monthly series keep the last observed value within each month. The result is a wide panel with one row per month and one column per project series name.

In [ ]:
panel = monthly_panel_from_long(raw, dictionary_df=series_dictionary)

display(panel.head())
print(f"Monthly panel shape: {panel.shape[0]:,} rows x {panel.shape[1]:,} columns")

## Construct key variables

This step creates the credit shares, accounting check variables, log credit stocks, monthly credit growth rates, policy-rate changes, optional exchange-rate changes, and the high-directed-share regime indicator used in the state-dependent specifications. The credit-stock identities are project assumptions, so they are built explicitly rather than hidden inside later estimation code.

In [ ]:
panel = add_credit_shares(panel)
panel = add_log_credit_variables(panel)
panel = add_growth_rates(panel)
panel = add_policy_variables(panel)
panel = add_state_variables(panel)

constructed_columns = [
    "directed_credit_share",
    "free_credit_share",
    "credit_gap_check",
    "credit_gap_check_pct",
    "log_credit_total_stock",
    "log_free_credit_stock",
    "log_directed_credit_stock",
    "growth_credit_total_stock",
    "growth_free_credit_stock",
    "growth_directed_credit_stock",
    "delta_selic",
    "exchange_rate_log_change",
    "high_directed_share",
]
display(panel[[col for col in constructed_columns if col in panel.columns]].head())

## Diagnostics

These diagnostics are used to catch coding or concept mistakes before the cleaned panel is used in figures or local projections. In particular, the credit-gap check should be small if the total, free, and directed credit stock concepts line up as expected.

In [ ]:
print(f"Panel date range: {panel['month'].min().date()} to {panel['month'].max().date()}")
print(f"Rows: {panel.shape[0]:,}")
print(f"Columns: {panel.shape[1]:,}")

print("Missing values by column:")
display(missing_summary(panel))

print("First and last non-missing date by column:")
display(first_last_nonmissing(panel, date_col="month"))

key_constructed_variables = [
    "directed_credit_share",
    "free_credit_share",
    "credit_gap_check_pct",
    "growth_credit_total_stock",
    "growth_free_credit_stock",
    "growth_directed_credit_stock",
    "delta_selic",
    "exchange_rate_log_change",
    "high_directed_share",
]
key_constructed_variables = [
    col for col in key_constructed_variables if col in panel.columns
]

print("Summary statistics for key constructed variables:")
display(panel[key_constructed_variables].describe().T)

print(
    "directed_credit_share min/max:",
    panel["directed_credit_share"].min(),
    panel["directed_credit_share"].max(),
)
print(
    "free_credit_share min/max:",
    panel["free_credit_share"].min(),
    panel["free_credit_share"].max(),
)
print(
    "credit_gap_check_pct mean absolute value:",
    panel["credit_gap_check_pct"].abs().mean(),
)
print(
    "credit_gap_check_pct max absolute value:",
    panel["credit_gap_check_pct"].abs().max(),
)

## Save cleaned panel

Finally, we save the cleaned monthly panel to `data/processed/`. This CSV is intentionally generated output: it can be recreated by rerunning the download and cleaning notebooks.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
panel.to_csv(OUTPUT_FILE, index=False)

print(f"Saved cleaned monthly panel to: {OUTPUT_FILE}")